In [2]:
import os, torch
from torch.utils.data import DataLoader
from tsfnet_model_notebook import TSFNet, NBackDataset, _unpack_batch

device = "cuda" if torch.cuda.is_available() else "cpu"
RESULTS_DIR = r"E:\kntu\Thesis\codes\Results\TSFNET_PAPER_EXACT_nback\results_fda0fd34fd_seed999"
SUBJECT_DIR_ROOT = r"E:\kntu\Thesis\codes\Results\TSFNET_PAPER_EXACT_nback"
TEST_SESSION = 1
subject_ids = [f"VP{i:03d}" for i in range(1, 27)]

In [3]:
from chance_level_permutation_test import trial_level_permutation_test
import numpy as np

all_true, all_pred, all_trial = [], [], []

for vp in subject_ids:
    subject_dir = os.path.join(SUBJECT_DIR_ROOT, vp)
    full = NBackDataset(subject_dir, vp)
    test_idx = [i for i, s in enumerate(full.session_ids) if s == TEST_SESSION]
    test_ds = NBackDataset(subject_dir, vp, indices=test_idx)
    test_trial_ids = full.trial_ids[test_idx]
    test_loader = DataLoader(test_ds, batch_size=16)

    ckpt_path = os.path.join(RESULTS_DIR, f"{vp}_session{TEST_SESSION}_best.pt")
    model = TSFNet(num_classes=3, cafe_d_model=64, fnirs_num_segments=21).to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval()

    preds, labels = [], []
    with torch.no_grad():
        for batch in test_loader:
            eeg_b, fnirs_b, lbl_b, _, _ = _unpack_batch(batch)
            _, _, _, y_pred, _ = model(eeg_b.to(device), fnirs_b.to(device))
            preds.append(y_pred.argmax(dim=1).cpu().numpy())
            labels.append(lbl_b.numpy())
    preds = np.concatenate(preds)
    labels = np.concatenate(labels)

    # trial_id رو global و منحصربه‌فرد بین آزمودنی‌ها می‌کنیم (طبق توصیه‌ی docstring تابع)
    global_trial_ids = [f"{vp}_{t}" for t in test_trial_ids]

    all_true.append(labels)
    all_pred.append(preds)
    all_trial.extend(global_trial_ids)

    del model
    torch.cuda.empty_cache()

all_true = np.concatenate(all_true)
all_pred = np.concatenate(all_pred)
all_trial = np.array(all_trial)

result = trial_level_permutation_test(all_true, all_pred, all_trial, n_permutations=10000, seed=42)
print("Observed accuracy:", result["observed_accuracy"])
print("p-value:", result["p_value"])
print("Empirical chance (95th pct):", result["chance_95th_percentile"])

Observed accuracy: 0.6614102564102564
p-value: 9.999000099990002e-05
Empirical chance (95th pct): 0.37808333333333327


In [4]:
# میانگین per-subject (قابل مقایسه مستقیم با ۶۱٫۲۰٪ گزارش‌شده)
per_subject_acc = []
start = 0
for vp in subject_ids:
    n = sum(1 for t in all_trial if t.startswith(vp + "_"))
    # اینجا باید از داده‌ای که تو حلقه‌ی قبلی per-subject جمع کردید استفاده کنید

In [5]:
per_subject_results = {}
# ... داخل حلقه‌ی for vp in subject_ids، بعد از محاسبه‌ی preds/labels:
per_subject_results[vp] = np.mean(preds == labels)

# بعد از پایان حلقه:
print("میانگین per-subject:", np.mean(list(per_subject_results.values())))
print("میانگین تجمیعی (پنجره‌محور):", np.mean(all_true == all_pred))

میانگین per-subject: 0.7333333333333333
میانگین تجمیعی (پنجره‌محور): 0.6614102564102564


In [6]:
print("تعداد آزمودنی‌های ثبت‌شده:", len(per_subject_results))
print(per_subject_results)


تعداد آزمودنی‌های ثبت‌شده: 1
{'VP026': np.float64(0.7333333333333333)}


In [7]:
from chance_level_permutation_test import trial_level_permutation_test
import numpy as np

all_true, all_pred, all_trial = [], [], []
per_subject_results = {}   # <-- این‌جا، بیرون و قبل از حلقه تعریف بشه

for vp in subject_ids:
    subject_dir = os.path.join(SUBJECT_DIR_ROOT, vp)
    full = NBackDataset(subject_dir, vp)
    test_idx = [i for i, s in enumerate(full.session_ids) if s == TEST_SESSION]
    test_ds = NBackDataset(subject_dir, vp, indices=test_idx)
    test_trial_ids = full.trial_ids[test_idx]
    test_loader = DataLoader(test_ds, batch_size=16)

    ckpt_path = os.path.join(RESULTS_DIR, f"{vp}_session{TEST_SESSION}_best.pt")
    model = TSFNet(num_classes=3, cafe_d_model=64, fnirs_num_segments=21).to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval()

    preds, labels = [], []
    with torch.no_grad():
        for batch in test_loader:
            eeg_b, fnirs_b, lbl_b, _, _ = _unpack_batch(batch)
            _, _, _, y_pred, _ = model(eeg_b.to(device), fnirs_b.to(device))
            preds.append(y_pred.argmax(dim=1).cpu().numpy())
            labels.append(lbl_b.numpy())
    preds = np.concatenate(preds)
    labels = np.concatenate(labels)

    per_subject_results[vp] = np.mean(preds == labels)   # <-- داخل حلقه، همین‌جا

    global_trial_ids = [f"{vp}_{t}" for t in test_trial_ids]
    all_true.append(labels)
    all_pred.append(preds)
    all_trial.extend(global_trial_ids)

    del model
    torch.cuda.empty_cache()

all_true = np.concatenate(all_true)
all_pred = np.concatenate(all_pred)
all_trial = np.array(all_trial)

print("تعداد آزمودنی‌های ثبت‌شده:", len(per_subject_results))
print("میانگین per-subject:", np.mean(list(per_subject_results.values())))
print("میانگین تجمیعی (پنجره‌محور):", np.mean(all_true == all_pred))

تعداد آزمودنی‌های ثبت‌شده: 26
میانگین per-subject: 0.6591760469154202
میانگین تجمیعی (پنجره‌محور): 0.6614102564102564


In [8]:
from chance_level_permutation_test import trial_level_permutation_test
from tsfnet_model_notebook import adapt_batchnorm_to_target
import numpy as np

all_true, all_pred, all_trial = [], [], []
per_subject_results = {}

for vp in subject_ids:
    subject_dir = os.path.join(SUBJECT_DIR_ROOT, vp)
    full = NBackDataset(subject_dir, vp)
    test_idx = [i for i, s in enumerate(full.session_ids) if s == TEST_SESSION]
    test_ds = NBackDataset(subject_dir, vp, indices=test_idx)
    test_trial_ids = full.trial_ids[test_idx]
    test_loader = DataLoader(test_ds, batch_size=16)

    ckpt_path = os.path.join(RESULTS_DIR, f"{vp}_session{TEST_SESSION}_best.pt")
    model = TSFNet(num_classes=3, cafe_d_model=64, fnirs_num_segments=21).to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval()

    # *** خط اضافه‌شده *** -- AdaBN (E4) رو دقیقاً مثل training دوباره اجرا می‌کنیم
    model = adapt_batchnorm_to_target(model, test_loader, device, log_fn=lambda *a, **k: None)

    preds, labels = [], []
    with torch.no_grad():
        for batch in test_loader:
            eeg_b, fnirs_b, lbl_b, _, _ = _unpack_batch(batch)
            _, _, _, y_pred, _ = model(eeg_b.to(device), fnirs_b.to(device))
            preds.append(y_pred.argmax(dim=1).cpu().numpy())
            labels.append(lbl_b.numpy())
    preds = np.concatenate(preds)
    labels = np.concatenate(labels)

    per_subject_results[vp] = np.mean(preds == labels)

    global_trial_ids = [f"{vp}_{t}" for t in test_trial_ids]
    all_true.append(labels)
    all_pred.append(preds)
    all_trial.extend(global_trial_ids)

    del model
    torch.cuda.empty_cache()

all_true = np.concatenate(all_true)
all_pred = np.concatenate(all_pred)
all_trial = np.array(all_trial)

print("تعداد آزمودنی‌ها:", len(per_subject_results))
print("میانگین per-subject:", np.mean(list(per_subject_results.values())))
print("میانگین تجمیعی (پنجره‌محور):", np.mean(all_true == all_pred))

تعداد آزمودنی‌ها: 26
میانگین per-subject: 0.6711165421362624
میانگین تجمیعی (پنجره‌محور): 0.6733333333333333


In [9]:
from chance_level_permutation_test import trial_level_permutation_test
from tsfnet_model_notebook import adapt_batchnorm_to_target
import numpy as np

all_true, all_pred, all_trial = [], [], []
per_subject_results = {}

for vp in subject_ids:
    subject_dir = os.path.join(SUBJECT_DIR_ROOT, vp)
    full = NBackDataset(subject_dir, vp)
    all_sessions = sorted(np.unique(full.session_ids))

    subj_preds, subj_labels = [], []

    for test_session in all_sessions:   # <-- حلقه‌ی جدید روی هر ۳ session
        test_idx = [i for i, s in enumerate(full.session_ids) if s == test_session]
        test_ds = NBackDataset(subject_dir, vp, indices=test_idx)
        test_trial_ids = full.trial_ids[test_idx]
        test_loader = DataLoader(test_ds, batch_size=16)

        ckpt_path = os.path.join(RESULTS_DIR, f"{vp}_session{test_session}_best.pt")
        model = TSFNet(num_classes=3, cafe_d_model=64, fnirs_num_segments=21).to(device)
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
        model.eval()
        model = adapt_batchnorm_to_target(model, test_loader, device, log_fn=lambda *a, **k: None)

        preds, labels = [], []
        with torch.no_grad():
            for batch in test_loader:
                eeg_b, fnirs_b, lbl_b, _, _ = _unpack_batch(batch)
                _, _, _, y_pred, _ = model(eeg_b.to(device), fnirs_b.to(device))
                preds.append(y_pred.argmax(dim=1).cpu().numpy())
                labels.append(lbl_b.numpy())
        preds = np.concatenate(preds)
        labels = np.concatenate(labels)

        global_trial_ids = [f"{vp}_S{test_session}_{t}" for t in test_trial_ids]
        all_true.append(labels)
        all_pred.append(preds)
        all_trial.extend(global_trial_ids)
        subj_preds.append(preds)
        subj_labels.append(labels)

        del model
        torch.cuda.empty_cache()

    per_subject_results[vp] = np.mean(np.concatenate(subj_preds) == np.concatenate(subj_labels))

all_true = np.concatenate(all_true)
all_pred = np.concatenate(all_pred)
all_trial = np.array(all_trial)

print("میانگین per-subject (هر ۳ session):", np.mean(list(per_subject_results.values())))
print("میانگین تجمیعی (پنجره‌محور):", np.mean(all_true == all_pred))

میانگین per-subject (هر ۳ session): 0.622007262749682
میانگین تجمیعی (پنجره‌محور): 0.6232449297971919


In [1]:
result = trial_level_permutation_test(all_true, all_pred, all_trial, n_permutations=10000, seed=42)
print("Observed accuracy:", result["observed_accuracy"])
print("p-value:", result["p_value"])
print("Empirical chance (mean):", result["chance_mean"])
print("Empirical chance (95th pct):", result["chance_95th_percentile"])
print("Empirical chance (99th pct):", result["chance_99th_percentile"])

NameError: name 'trial_level_permutation_test' is not defined

In [ ]:
#این اصلیه

In [2]:
# سلول ۱: setup
import os, torch
from torch.utils.data import DataLoader
from tsfnet_model_notebook import TSFNet, NBackDataset, _unpack_batch, adapt_batchnorm_to_target
from chance_level_permutation_test import trial_level_permutation_test
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
RESULTS_DIR = r"E:\kntu\Thesis\codes\Results\TSFNET_PAPER_EXACT_nback\results_fda0fd34fd_seed999"
SUBJECT_DIR_ROOT = r"E:\kntu\Thesis\codes\Results\TSFNET_PAPER_EXACT_nback"
subject_ids = [f"VP{i:03d}" for i in range(1, 27)]
# سلول ۲: جمع‌آوری پیش‌بینی‌ها (همون کد قبلی که درست جواب داد)
all_true, all_pred, all_trial = [], [], []
per_subject_results = {}

for vp in subject_ids:
    subject_dir = os.path.join(SUBJECT_DIR_ROOT, vp)
    full = NBackDataset(subject_dir, vp)
    all_sessions = sorted(np.unique(full.session_ids))

    subj_preds, subj_labels = [], []

    for test_session in all_sessions:
        test_idx = [i for i, s in enumerate(full.session_ids) if s == test_session]
        test_ds = NBackDataset(subject_dir, vp, indices=test_idx)
        test_trial_ids = full.trial_ids[test_idx]
        test_loader = DataLoader(test_ds, batch_size=16)

        ckpt_path = os.path.join(RESULTS_DIR, f"{vp}_session{test_session}_best.pt")
        model = TSFNet(num_classes=3, cafe_d_model=64, fnirs_num_segments=21).to(device)
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
        model.eval()
        model = adapt_batchnorm_to_target(model, test_loader, device, log_fn=lambda *a, **k: None)

        preds, labels = [], []
        with torch.no_grad():
            for batch in test_loader:
                eeg_b, fnirs_b, lbl_b, _, _ = _unpack_batch(batch)
                _, _, _, y_pred, _ = model(eeg_b.to(device), fnirs_b.to(device))
                preds.append(y_pred.argmax(dim=1).cpu().numpy())
                labels.append(lbl_b.numpy())
        preds = np.concatenate(preds)
        labels = np.concatenate(labels)

        global_trial_ids = [f"{vp}_S{test_session}_{t}" for t in test_trial_ids]
        all_true.append(labels)
        all_pred.append(preds)
        all_trial.extend(global_trial_ids)
        subj_preds.append(preds)
        subj_labels.append(labels)

        del model
        torch.cuda.empty_cache()

    per_subject_results[vp] = np.mean(np.concatenate(subj_preds) == np.concatenate(subj_labels))

all_true = np.concatenate(all_true)
all_pred = np.concatenate(all_pred)
all_trial = np.array(all_trial)

print("میانگین per-subject:", np.mean(list(per_subject_results.values())))
print("میانگین تجمیعی:", np.mean(all_true == all_pred))

میانگین per-subject: 0.622007262749682
میانگین تجمیعی: 0.6232449297971919


In [3]:
# سلول ۲: جمع‌آوری پیش‌بینی‌ها (همون کد قبلی که درست جواب داد)
all_true, all_pred, all_trial = [], [], []
per_subject_results = {}

for vp in subject_ids:
    subject_dir = os.path.join(SUBJECT_DIR_ROOT, vp)
    full = NBackDataset(subject_dir, vp)
    all_sessions = sorted(np.unique(full.session_ids))

    subj_preds, subj_labels = [], []

    for test_session in all_sessions:
        test_idx = [i for i, s in enumerate(full.session_ids) if s == test_session]
        test_ds = NBackDataset(subject_dir, vp, indices=test_idx)
        test_trial_ids = full.trial_ids[test_idx]
        test_loader = DataLoader(test_ds, batch_size=16)

        ckpt_path = os.path.join(RESULTS_DIR, f"{vp}_session{test_session}_best.pt")
        model = TSFNet(num_classes=3, cafe_d_model=64, fnirs_num_segments=21).to(device)
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
        model.eval()
        model = adapt_batchnorm_to_target(model, test_loader, device, log_fn=lambda *a, **k: None)

        preds, labels = [], []
        with torch.no_grad():
            for batch in test_loader:
                eeg_b, fnirs_b, lbl_b, _, _ = _unpack_batch(batch)
                _, _, _, y_pred, _ = model(eeg_b.to(device), fnirs_b.to(device))
                preds.append(y_pred.argmax(dim=1).cpu().numpy())
                labels.append(lbl_b.numpy())
        preds = np.concatenate(preds)
        labels = np.concatenate(labels)

        global_trial_ids = [f"{vp}_S{test_session}_{t}" for t in test_trial_ids]
        all_true.append(labels)
        all_pred.append(preds)
        all_trial.extend(global_trial_ids)
        subj_preds.append(preds)
        subj_labels.append(labels)

        del model
        torch.cuda.empty_cache()

    per_subject_results[vp] = np.mean(np.concatenate(subj_preds) == np.concatenate(subj_labels))

all_true = np.concatenate(all_true)
all_pred = np.concatenate(all_pred)
all_trial = np.array(all_trial)

print("میانگین per-subject:", np.mean(list(per_subject_results.values())))
print("میانگین تجمیعی:", np.mean(all_true == all_pred))

میانگین per-subject: 0.622007262749682
میانگین تجمیعی: 0.6232449297971919


In [4]:
result = trial_level_permutation_test(all_true, all_pred, all_trial, n_permutations=10000, seed=42)
print("Observed accuracy:", result["observed_accuracy"])
print("p-value:", result["p_value"])
print("Empirical chance (mean):", result["chance_mean"])
print("Empirical chance (95th pct):", result["chance_95th_percentile"])
print("Empirical chance (99th pct):", result["chance_99th_percentile"])

Observed accuracy: 0.6232449297971919
p-value: 9.999000099990002e-05
Empirical chance (mean): 0.33312438464205235
Empirical chance (95th pct): 0.3587298491939677
Empirical chance (99th pct): 0.369258537008147
